# Fine-tune FLAN-T5 for CBRN emergency preparedness

This notebook fine-tunes a small pretrained transformer on protective public-health guidance. It excludes agent creation, weaponization, harmful dispersal, optimization, and evasion content.

Choose **Runtime → Change runtime type → T4 GPU**, then run each cell in order.

In [ ]:
%pip install -q "transformers>=4.40,<6" sentencepiece

Upload `finetune_cbrn.py` and `cbrn_preparedness.jsonl` from the `examples/cbrn_finetuning` folder.

In [ ]:
from google.colab import files

uploaded = files.upload()
required = {"finetune_cbrn.py", "cbrn_preparedness.jsonl"}
missing = required - set(uploaded)
if missing:
    raise ValueError(f"Missing files: {', '.join(sorted(missing))}")

## Train and evaluate

The script records answers from the original model, fine-tunes for twenty epochs, evaluates the same held-out set again, and saves both versions in the report.

In [ ]:
!python finetune_cbrn.py --data cbrn_preparedness.jsonl --epochs 20 --batch-size 8 --output cbrn-flan-t5

## Compare the results

In [ ]:
import json
import pandas as pd

with open("cbrn-flan-t5/evaluation.json") as file:
    report = json.load(file)

print("Baseline metrics:", report["before"])
print("Fine-tuned metrics:", report["after"])
pd.DataFrame(report["predictions"])[
    ["category", "question", "baseline_answer", "fine_tuned_answer"]
]

## Download the trained model and evaluation

In [ ]:
import shutil

shutil.make_archive("cbrn-flan-t5", "zip", "cbrn-flan-t5")
files.download("cbrn-flan-t5.zip")